In [1]:
# ============================================================================
# 2_score.ipynb  —  All model scoring (judge, metrics, robustness), persisted
# ----------------------------------------------------------------------------
# MERGES former 03 (judge + proxy faithfulness), 03c (production faithfulness,
# PATCHED), 03d (judge-model robustness), 06 (construct validity), 03b
# (inter-judge). This is the ONLY place that spends API budget besides notebook 1.
# It writes raw score parquets; all statistics/tables are computed offline in
# notebook 3, all figures in notebook 4.
#
# SPEED: API calls run concurrently (MAX_WORKERS threads), and faithfulness is
# computed only on gap_given rows (the only ones any analysis consumes). Judge
# still runs on both generation conditions (the 2x2 ablation needs gap_blind).
#
# Toggle the blocks below to stage cost. Each writes its own parquet.
#
# INPUT   data/perturbed.parquet, data/with_override.parquet
# OUTPUT  data/scored.parquet           (judge informed/blind + proxy + production faithfulness)
#         data/judge_models.parquet     (multi-model judge on P0/P3/P4)
#         data/construct_scored.parquet (P0/P1/P4/P4b x grade shown/hidden)
#         data/interjudge.parquet       (secondary-judge re-score sample)
# ============================================================================
import sys, os, json, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd

_HERE = Path.cwd()
_ROOT = _HERE if (_HERE / "src").exists() else _HERE.parent
sys.path.insert(0, str(_ROOT / "src"))
from common import (PROJECT_ROOT, DATA_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR,
                    load_env, set_grayscale_style, save_fig, save_table,
                    require_input, RANDOM_SEED)
import matplotlib.pyplot as plt
import seaborn as sns
set_grayscale_style()
np.random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore")

cfg = load_env()
from openai import OpenAI
client = OpenAI(api_key=cfg["api_key"])

JUDGE_PRIMARY = cfg["strong"]
JUDGE_SECONDARY = cfg["mid"]
FAITH_MODEL = cfg["mid"]

# ---- which blocks to run (set False to skip an expensive block) --------------
RUN_JUDGE_PROXY = True           # core: judge + proxy faithfulness -> scored.parquet
RUN_PRODUCTION_FAITH = True      # PATCHED production faithfulness -> adds faithfulness_rag
RUN_JUDGE_MODELS = True          # multi-judge-model robustness
RUN_CONSTRUCT = True             # construct-validity (P4b + grade shown/hidden)
RUN_INTERJUDGE = True            # secondary-judge agreement sample
JUDGE_MODELS = list(dict.fromkeys([m for m in [cfg["weak"], cfg["mid"], cfg["strong"]] if m]))
INTERJUDGE_N = 60
CONSTRUCT_NOTCH = 2
MDNA_BUDGET = 6000
MAX_WORKERS = 8                  # concurrent API calls. Lower if you hit rate limits.

def tmap(fn, items):
    """Threaded map preserving input order (I/O-bound API calls)."""
    items = list(items)
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        return list(ex.map(fn, items))

RATING_ORDER = ["D", "C", "CC", "CCC", "B", "BB", "BBB", "A", "AA", "AAA"]
idx = {g: i for i, g in enumerate(RATING_ORDER)}
def shift_grade(g, k):
    if g not in idx: return g
    return RATING_ORDER[max(0, min(len(RATING_ORDER) - 1, idx[g] + k))]

pert = pd.read_parquet(require_input(DATA_DIR / "perturbed.parquet",
                                     hint="Run notebook 1 first."))
ov = pd.read_parquet(require_input(DATA_DIR / "with_override.parquet",
                                   hint="Run notebook 0 first."))

# ---- per-case fixed context: financial profile (+ MD&A) for faithfulness -----
non_feat = {"y_rank", "yhat", "lo", "hi", "delta", "override", "rating_year",
            "cik", "filing_year", "target_filing_year", "mdna_len", "year_offset"}
feat_cols = [c for c in ov.columns
             if pd.api.types.is_numeric_dtype(ov[c]) and c not in non_feat]
_ctx = {}
for _, r in ov.iterrows():
    key = (r.cik, int(r.rating_year))
    feats = "; ".join(f"{c}={float(r[c]):.4g}" for c in feat_cols if pd.notna(r.get(c)))
    mdna = str(r.get("mdna_text", ""))[:MDNA_BUDGET]
    _ctx[key] = (f"Sector: {r.get('Sector','NA')}. Financial profile: {feats}.", mdna)

def case_context(cik, rating_year):
    return _ctx.get((cik, int(rating_year)), ("Financial profile unavailable.", ""))

# ---- judge (five axes) -------------------------------------------------------
def judge_axes(rationale, rating, delta, informed, model, show_grade=True):
    gap_ctx = (f"Context: the agency rating is {int(delta):+d} notches vs the "
               f"quantitative profile; a good rationale presents a qualitative "
               f"justification whose direction matches this gap." if informed else
               "Score the rationale on its own internal validity.")
    sys_msg = ("You are evaluating a credit rating rationale. Score five axes "
               "1-5: grade_accuracy, grade_rationale_consistency, "
               "qualitative_adequacy, factual_faithfulness, explainability. "
               "qualitative_adequacy: 5=clear & consistent with the gap's "
               "direction, 3=present but generic, 1=absent or contradicting.")
    grade_line = (f"Rating: {rating}\n" if show_grade else
                  "Rating: [withheld — score the rationale's reasoning only]\n")
    user = f"{gap_ctx}\n{grade_line}Rationale:\n{rationale}\n\nReturn JSON with the five integer axes."
    r = client.chat.completions.create(model=model, temperature=0.0,
        messages=[{"role": "system", "content": sys_msg},
                  {"role": "user", "content": user}],
        response_format={"type": "json_object"})
    return json.loads(r.choices[0].message.content)

# ---- proxy faithfulness (paper's ORIGINAL metric: rationale-only, no profile
# passed). Passing the terse numeric profile floors this metric, so we keep the
# published behaviour exactly; `profile` is accepted but intentionally unused.
def faithfulness_proxy(rationale, profile=None):
    sys_msg = ("Return JSON {\"entailed\": int, \"total\": int}: of the "
               "quantitative/factual claims in the rationale, how many are "
               "entailed by the stated financial sector/profile.")
    r = client.chat.completions.create(model=FAITH_MODEL, temperature=0.0,
        messages=[{"role": "system", "content": sys_msg},
                  {"role": "user", "content": str(rationale)[:4000]}],
        response_format={"type": "json_object"})
    o = json.loads(r.choices[0].message.content)
    return int(o.get("entailed", 0)) / max(1, int(o.get("total", 1)))

# ---- PATCHED production faithfulness: decomposition restricted to QUANTITATIVE
# claims, verified against the financial profile (+ MD&A as supporting context).
# Scoping to quantitative claims (a) restores dynamic range -- P3 factual errors
# are caught -- and (b) keeps the metric blind to P4, whose reversal touches only
# QUALITATIVE support direction, so no quantitative claim changes. This fixes the
# flooring seen when ALL atomic claims (incl. unverifiable qualitative ones) were
# extracted and marked 0.
def extract_quant_claims(rationale):
    sys_msg = ("Extract ONLY the QUANTITATIVE or hard FACTUAL claims from the text "
               "(specific numbers, ratios, financial magnitudes, or directional "
               "statements about the financials). EXCLUDE purely qualitative or "
               "subjective statements (management quality, outlook tone, strategy). "
               "Return JSON {\"claims\": [<str>, ...]}; [] if none.")
    r = client.chat.completions.create(model=FAITH_MODEL, temperature=0.0,
        messages=[{"role": "system", "content": sys_msg},
                  {"role": "user", "content": str(rationale)[:4000]}],
        response_format={"type": "json_object"})
    cl = json.loads(r.choices[0].message.content).get("claims", [])
    return [c for c in cl if isinstance(c, str) and c.strip()]

def verify_quant(claims, profile, mdna=None):
    if not claims:
        return 1.0        # vacuously faithful
    sys_msg = ("You verify factual entailment of QUANTITATIVE claims. For each "
               "claim decide SUPPORTED by the CONTEXT (1) or NOT supported / "
               "contradicted (0). Judge only against the context; unverifiable=0. "
               "Return JSON {\"verdicts\":[<0|1>,...]} aligned to the claims order.")
    # Verify against the fixed financial PROFILE ONLY. Including the MD&A made the
    # verifier rubber-stamp everything (high-flat, no P3 sensitivity); the profile
    # lets an injected contradicting number (P3) be caught while a reversed
    # QUALITATIVE support argument (P4) changes no quantitative claim -> stays blind.
    context = profile
    user = "CONTEXT:\n" + context[:8000] + "\n\nCLAIMS:\n" + json.dumps(claims, ensure_ascii=False)
    r = client.chat.completions.create(model=FAITH_MODEL, temperature=0.0,
        messages=[{"role": "system", "content": sys_msg},
                  {"role": "user", "content": user}],
        response_format={"type": "json_object"})
    v = [int(x) for x in json.loads(r.choices[0].message.content).get("verdicts", [])
         if x in (0, 1, "0", "1")][:len(claims)]
    return (sum(v) / len(claims)) if v else 0.0

# ============================================================================
# BLOCK 1 — scored.parquet : judge (informed+blind) + proxy + production faith
# ============================================================================
if RUN_JUDGE_PROXY:
    def score_row(r):
        profile, mdna = case_context(r["cik"], r["rating_year"])
        # Faithfulness is consumed ONLY on gap_given outputs (all divergence /
        # decision-impact analysis filters condition=="gap_given"), so we skip it
        # on gap_blind rows to avoid wasted calls. Judge runs on both conditions
        # (the 2x2 ablation needs gap_blind judging).
        if r["condition"] == "gap_given":
            try: fp = faithfulness_proxy(r["rationale"], profile)
            except Exception: fp = np.nan
            if RUN_PRODUCTION_FAITH:
                try: fr = verify_quant(extract_quant_claims(r["rationale"]), profile, mdna)
                except Exception: fr = np.nan
            else:
                fr = np.nan
        else:
            fp = fr = np.nan
        out = []
        for informed in (True, False):
            try:
                ax = judge_axes(r["rationale"], r["rating"], r["delta"], informed, JUDGE_PRIMARY)
                jq = ax.get("qualitative_adequacy", np.nan)
            except Exception:
                ax, jq = {}, np.nan
            out.append({"case_id": r["case_id"], "condition": r["condition"],
                        "pert": r["pert"], "k": r["k"],
                        "judge_cond": "informed" if informed else "blind",
                        "rating": r["rating"], "rationale": r["rationale"], "delta": int(r["delta"]),
                        "judge_qual": jq, "judge_axes": json.dumps(ax),
                        "faithfulness": fp, "faithfulness_rag": fr})
        return out
    results = tmap(score_row, pert.to_dict("records"))
    scored = pd.DataFrame([row for sub in results for row in sub])
    scored.to_parquet(DATA_DIR / "scored.parquet", index=False)
    print("[block1] scored.parquet rows:", len(scored))

# ============================================================================
# BLOCK 2 — judge_models.parquet : P0/P3/P4 across a list of judge models
# ============================================================================
if RUN_JUDGE_MODELS:
    work = pert[(pert.condition == "gap_given") & (pert.pert.isin(["P0", "P3", "P4"]))].to_dict("records")
    tasks = [(m, r) for m in JUDGE_MODELS for r in work]
    def jm_one(mr):
        m, r = mr
        try:
            q = judge_axes(r["rationale"], r["rating"], r["delta"], True, m).get("qualitative_adequacy", np.nan)
        except Exception:
            q = np.nan
        return {"judge_model": m, "case_id": r["case_id"], "pert": r["pert"], "judge_qual": q}
    jm_rows = tmap(jm_one, tasks)
    pd.DataFrame(jm_rows).to_parquet(DATA_DIR / "judge_models.parquet", index=False)
    print("[block2] judge_models.parquet rows:", len(jm_rows))

# ============================================================================
# BLOCK 3 — construct_scored.parquet : P0/P1/P4/P4b x grade shown/hidden
# ============================================================================
if RUN_CONSTRUCT:
    base = pert[(pert.condition == "gap_given")].copy()
    p4 = base[base.pert == "P4"][["case_id", "cik", "rating_year", "delta", "rating", "rationale"]].copy()
    p4["pert"] = "P4b"
    p4["rating"] = [shift_grade(g, -int(np.sign(d)) * CONSTRUCT_NOTCH)
                    for g, d in zip(p4["rating"], p4["delta"])]
    variants = pd.concat([base[base.pert.isin(["P0", "P1", "P4"])], p4], ignore_index=True)
    variants = variants[~((variants.pert == "P1") & (variants.k != 3))]
    tasks = [(r, show) for r in variants.to_dict("records") for show in (True, False)]
    def cs_one(rs):
        r, show = rs
        try:
            q = judge_axes(r["rationale"], r["rating"], r["delta"], True, JUDGE_PRIMARY,
                           show_grade=show).get("qualitative_adequacy", np.nan)
        except Exception:
            q = np.nan
        return {"case_id": r["case_id"], "pert": r["pert"],
                "grade_view": "shown" if show else "hidden", "judge_qual": q}
    cs_rows = tmap(cs_one, tasks)
    pd.DataFrame(cs_rows).to_parquet(DATA_DIR / "construct_scored.parquet", index=False)
    print("[block3] construct_scored.parquet rows:", len(cs_rows))

# ============================================================================
# BLOCK 4 — interjudge.parquet : secondary judge re-scores a random sample
# ============================================================================
if RUN_INTERJUDGE and RUN_JUDGE_PROXY:
    s = pd.read_parquet(DATA_DIR / "scored.parquet")
    pool = s[(s.judge_cond == "informed") & (s.condition == "gap_given")].dropna(subset=["judge_qual"])
    samp = pool.sample(n=min(INTERJUDGE_N, len(pool)), random_state=RANDOM_SEED).copy()
    def ij_one(r):
        try:
            return judge_axes(r["rationale"], r["rating"], int(r["delta"]), True,
                              JUDGE_SECONDARY).get("qualitative_adequacy", np.nan)
        except Exception:
            return np.nan
    samp["judge_qual_secondary"] = tmap(ij_one, samp.to_dict("records"))
    samp.to_parquet(DATA_DIR / "interjudge.parquet", index=False)
    print("[block4] interjudge.parquet rows:", len(samp))

print("\n[done] scoring complete. Run notebook 3 for tables, notebook 4 for figures.")


[block1] scored.parquet rows: 3648
[block2] judge_models.parquet rows: 1026
[block3] construct_scored.parquet rows: 912
[block4] interjudge.parquet rows: 60

[done] scoring complete. Run notebook 3 for tables, notebook 4 for figures.
